# Gradio Introduction

**Week 2 Day 2 - Learning Lab**

Building simple UIs for LLM applications with Gradio.

## Intent

Learn to:
- Create basic Gradio interfaces quickly
- Implement streaming responses with generators
- Build multi-model UIs with class-based design
- Use different component types (Textbox, Dropdown, Markdown)
- Share and deploy Gradio apps

## Expected Insights

- Gradio is perfect for demos, prototypes, and MVPs
- Streaming requires generator pattern (`yield` not `return`)
- Class-based design with model registry simplifies multi-model UIs
- Pattern: Use Gradio for fast prototyping, custom web for production


In [ ]:
# Setup
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


## Experiment 1: Basic Gradio Interface

Create a simple text input → LLM → text output interface.

**Key Pattern:** `gr.Interface(fn=function, inputs=[...], outputs=[...])`


In [ ]:
# Basic Gradio interface
def simple_llm(prompt):
    """Simple LLM call - returns complete response."""
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    return response.choices[0].message.content

# Create interface
interface = gr.Interface(
    fn=simple_llm,
    inputs=gr.Textbox(label="Your message:", lines=5),
    outputs=gr.Textbox(label="Response:", lines=10),
    title="Simple LLM Chat",
    examples=["Hello!", "Explain transformers"],
    flagging_mode="never"
)

# Launch (comment out to avoid auto-launching)
# interface.launch()


## Experiment 2: Streaming with Generators

**Key Pattern:** Use `yield` keyword for streaming responses.

**Why:** Better UX - users see responses as they generate, not all at once.


In [ ]:
# Streaming LLM with generator pattern
def stream_llm(prompt):
    """Stream LLM response - yields incremental updates."""
    messages = [{"role": "user", "content": prompt}]
    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True  # Enable streaming
    )
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result  # Yield accumulated result (not return!)

# Create streaming interface
streaming_interface = gr.Interface(
    fn=stream_llm,
    inputs=gr.Textbox(label="Your message:", lines=5),
    outputs=gr.Markdown(label="Response:"),  # Markdown for formatted output
    title="Streaming LLM Chat",
    examples=["Explain transformers", "Write a haiku"],
    flagging_mode="never"
)

# Launch (comment out to avoid auto-launching)
# streaming_interface.launch()


## Experiment 3: Multi-Model UI with Class-Based Design

**Key Pattern:** Model registry pattern - dictionary mapping model names to (client, model_name) tuples.

**Why:** Cleaner than separate functions per model, easy to extend.


In [ ]:
import requests

class MultiModelChat:
    """Unified chat interface supporting multiple LLM providers."""
    
    def __init__(self):
        """Initialize all model clients."""
        # OpenAI client
        self.openai_client = OpenAI()
        
        # Ollama client (check if available)
        self.ollama_available = False
        try:
            requests.get("http://localhost:11434/", timeout=2)
            self.ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
            self.ollama_available = True
        except:
            self.ollama_client = None
        
        # Model registry: maps display name to (client, model_name) tuple
        self.models = {
            "GPT": (self.openai_client, "gpt-4.1-mini"),
        }
        
        if self.ollama_available:
            self.models["Ollama"] = (self.ollama_client, "llama3.2")
    
    def chat(self, prompt, model_name):
        """
        Chat with selected model - streaming support.
        
        Args:
            prompt: User message
            model_name: Display name of model (e.g., "GPT", "Ollama")
        
        Yields:
            str: Incremental response chunks
        """
        if model_name not in self.models:
            yield f"Error: Model '{model_name}' not available."
            return
        
        client, model = self.models[model_name]
        
        messages = [{"role": "user", "content": prompt}]
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        
        result = ""
        for chunk in stream:
            result += chunk.choices[0].delta.content or ""
            yield result
    
    def get_available_models(self):
        """Return list of available model names."""
        return list(self.models.keys())


# Initialize chat
chat = MultiModelChat()

# Create multi-model interface
multi_model_interface = gr.Interface(
    fn=chat.chat,
    inputs=[
        gr.Textbox(label="Your message:", lines=5),
        gr.Dropdown(chat.get_available_models(), label="Select model", value="GPT")
    ],
    outputs=gr.Markdown(label="Response:"),
    title="Multi-Model Chat",
    examples=[
        ["Explain transformers", "GPT"],
        ["Write a haiku", "GPT"]
    ],
    flagging_mode="never"
)

# Launch (comment out to avoid auto-launching)
# multi_model_interface.launch()


## Key Takeaways

### Gradio Basics
- **Simple interface:** `gr.Interface(fn=function, inputs=[...], outputs=[...])`
- **Component types:** Textbox (text), Dropdown (selection), Markdown (formatted output)
- **Launch options:** `share=True` (public link), `inbrowser=True` (auto-open), `auth=("user", "pass")` (password)
- **Examples:** Pre-populate UI with example inputs

### Streaming Pattern
- **Generator function:** Must use `yield`, not `return`
- **Gradio auto-detection:** Gradio automatically detects generator functions
- **Accumulate and yield:** Build result incrementally, yield after each chunk
- **Better UX:** Users see responses as they generate

### Class-Based Multi-Model Design
- **Model registry:** Dictionary mapping names to (client, model_name) tuples
- **Unified method:** Single function works for all models in registry
- **Automatic detection:** Check availability, only include if ready
- **Easy extension:** Add new models by updating dictionary
- **Benefits:** No code duplication, cleaner architecture, self-documenting

### When to Use Gradio
- **Perfect for:** Demos, prototypes, MVPs, internal tools
- **Not ideal for:** Production apps requiring full customization
- **Pattern:** Use Gradio for fast prototyping, custom web for production
